# 0A — CSV to Parquet Conversion

Convert the raw Kaggle CSVs to parquet for faster I/O the rest of the week.  
Run this once, then never touch the CSVs again.

In [ ]:
import pandas as pd
from pathlib import Path
import time

# -------------------------------------------------------------------
# CONFIGURE THIS: point to wherever you unzipped the Kaggle download
# -------------------------------------------------------------------
RAW_DIR = Path("data/raw")          # contains articles.csv, customers.csv, transactions_train.csv
OUT_DIR = Path("data/parquet")
OUT_DIR.mkdir(parents=True, exist_ok=True)

: 

## Articles

In [ ]:
t0 = time.time()

articles = pd.read_csv(
    RAW_DIR / "articles.csv",
    dtype={"article_id": str},  # keep leading zeros
)

print(f"articles: {articles.shape}  ({time.time()-t0:.1f}s)")
articles.head()

In [ ]:
articles.to_parquet(OUT_DIR / "articles.parquet", index=False)
print("✓ articles.parquet saved")

## Customers

In [ ]:
t0 = time.time()

customers = pd.read_csv(
    RAW_DIR / "customers.csv",
    dtype={"customer_id": str},
)

print(f"customers: {customers.shape}  ({time.time()-t0:.1f}s)")
print(f"\nNull counts:\n{customers.isnull().sum()}")
customers.head()

In [ ]:
customers.to_parquet(OUT_DIR / "customers.parquet", index=False)
print("✓ customers.parquet saved")

## Transactions

This is the big one (~31M rows). We parse the date column upfront and enforce string dtypes on IDs.

In [ ]:
t0 = time.time()

transactions = pd.read_csv(
    RAW_DIR / "transactions_train.csv",
    dtype={"customer_id": str, "article_id": str},
    parse_dates=["t_dat"],
)

print(f"transactions: {transactions.shape}  ({time.time()-t0:.1f}s)")
print(f"Date range: {transactions['t_dat'].min()} → {transactions['t_dat'].max()}")
transactions.head()

In [ ]:
transactions.to_parquet(OUT_DIR / "transactions.parquet", index=False)
print("✓ transactions.parquet saved")

## Quick sanity check: reload from parquet and compare load times

In [ ]:
t0 = time.time()
_ = pd.read_parquet(OUT_DIR / "transactions.parquet")
print(f"Parquet reload: {time.time()-t0:.1f}s  (vs CSV above)")